# Real-Time Causal 6-Hour Sepsis Forecasting Neural Network & Monitoring Pipeline
**Project:** Local RAG-Powered Sepsis Clinical Decision Support (CDS)  
**Target Definition:** 6-hour forward Sepsis-3 onset forecast (`SepsisLabel = 1` starting at `t_sepsis - 6h` using strictly causal bedside history `<= hour t`).

### Architecture & Pipeline Overview
1. **Real-Time Bedside Data Alignment (`sepsis_realtime_cohort.zip`):** Uses the **chronological raw-unit hourly telemetry stream (`3,998+ ICU patients, ~155,000+ hourly ticks`)** matching live JSON streaming in `app/js/predictiveModel.js`.
2. **Causal Streaming Features (`79` -> `262D PLE` Representation):** Combines raw vitals/labs, **patient-personalized expanding baseline deviations** (`HR_diff_base`, `MAP_diff_base`, `Temp_diff_base`, `Resp_diff_base`), sliding vital trajectories (`1h/6h/12h`), clinical abnormality hinges, lab-update cadence, explicit missingness flags (for the 20 raw vitals/labs), signed-log1p compression, and **non-missing Quantile Piecewise-Linear Encodings (`PLE`)** across 28 core predictors.
3. **Imbalance-Aware 5-Fold PyTorch `PLE-ResNet` + Two-Tier Clinical Policy:** Handles natural `~43:1` ICU hourly class imbalance via capped inverse-prevalence **`pos_weight` + Pairwise AUROC Ranking Loss** in `Train`, keeps `Val` & `Test` at true incidence for **1D Platt Sigmoid Calibration**, and locks **Two Clinical Tiers** on `Val` (`Watchlist` Utility Screen + **`Critical Actionable Alert <= 5.0 FP/100h` with `2-of-3h` Sustained Episode Confirmation**).
4. **Deployment Monitoring Suite:** Tracks Hourly & Patient-Episode AUROC (with 95% Bootstrap CIs), Hourly & Patient-Episode AUPRC with Prevalence Baselines, Quantile Decile Calibration (`Brier`, `ECE`), Hourly & `2-of-3h` Patient Confusion Matrices, Early Lead Time, Subgroups, Train-vs-Test PSI Drift, and Neural Saliency / Permutation Importance.


## 1. Environment Setup & Kaggle Dataset Discovery
Imports PyTorch, scikit-learn, and calibration primitives, configures deterministic seeds/device (`CUDA` if available on Kaggle else `CPU`), and scans `/kaggle/input` for `sepsis_realtime_hourly.csv` / `.zip` / `.psv`.


In [ ]:
import ast, bisect, glob, json, math, os, time, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    precision_recall_curve, confusion_matrix, f1_score, brier_score_loss
)

torch.manual_seed(42)
np.random.seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

sns.set_theme(style="whitegrid")
INPUT_ROOT = "/kaggle/input" if os.path.exists("/kaggle/input") else "."
CUSTOM_INPUT_PATH = ""  # Optional: paste copied path from Kaggle right-hand Input panel (copy icon)
OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "data/kaggle_export"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEARCH_ROOTS = []
for cand_root in [CUSTOM_INPUT_PATH, INPUT_ROOT, "../input", "/kaggle/input/datasets"]:
    if cand_root and os.path.exists(cand_root) and cand_root not in SEARCH_ROOTS:
        SEARCH_ROOTS.append(cand_root)

print("PyTorch:", torch.__version__, "| Device:", DEVICE, "| Search roots:", SEARCH_ROOTS)
if os.path.exists("/kaggle/input"):
    print("Top-level /kaggle/input entries:", sorted(os.listdir("/kaggle/input")))

scanned_dirs = 0
for s_root in SEARCH_ROOTS:
    for root, _, files in os.walk(s_root, followlinks=True):
        if files and not any(skip in root for skip in (".git", "node_modules", ".venv", "scratch")):
            scanned_dirs += 1
            if scanned_dirs <= 30:
                print(f"  {root} -> {len(files)} files (sample: {sorted(files)[:3]})")
if scanned_dirs == 0 and os.path.exists("/kaggle/input"):
    print("WARNING: No files visible under /kaggle/input in this kernel session! If you just added/uploaded the dataset, click Session -> Restart Session (or Refresh) in Kaggle, then re-run from Cell 1.")


### Dataset Discovery Check
Verify that `sepsis_realtime_hourly.csv`, `sepsis_realtime_cohort.zip`, or `.psv` files appear above before running Cell 2. If `Top-level /kaggle/input entries: []` appears, click **Restart Session** in Kaggle or paste the copied folder path from the right-hand Input panel into `CUSTOM_INPUT_PATH` above.


## 2. Adaptive Real-Time Stream Loader & Scale Audit
Scans mounted inputs (including symlinked dataset directories and `raw_psv/` subfolders, deduplicating `patient_id`), prioritizing `sepsis_realtime_hourly.csv` / `sepsis_realtime_cohort.zip` and falling back cleanly to `.psv` trajectories or `.zip` archives.


In [ ]:
def load_realtime_icu_cohort(input_root):
    roots = [r for r in [CUSTOM_INPUT_PATH, input_root, "../input", "/kaggle/input/datasets"] if r and os.path.exists(r)]
    all_files = []
    seen_real = set()
    for r in roots:
        for dirpath, _, filenames in os.walk(r, followlinks=True):
            if any(skip in dirpath for skip in (".git", "node_modules", ".venv", "scratch")):
                continue
            for fn in filenames:
                fp = os.path.join(dirpath, fn)
                rp = os.path.realpath(fp)
                if rp not in seen_real:
                    seen_real.add(rp)
                    all_files.append(fp)

    # Priority 1: Explicit real-time longitudinal CSV
    rt_csvs = sorted(f for f in all_files if "sepsis_realtime_hourly" in os.path.basename(f).lower() and f.lower().endswith(".csv"))
    if rt_csvs:
        df = pd.read_csv(rt_csvs[0])
        print(f"Loaded real-time cohort CSV: {rt_csvs[0]} -> {len(df):,} rows, {df['patient_id'].nunique():,} patients")
        return df

    # Priority 2: Zipped real-time export if uploaded directly
    rt_zips = sorted(f for f in all_files if "sepsis_realtime_cohort" in os.path.basename(f).lower() and f.lower().endswith(".zip"))
    if rt_zips:
        with zipfile.ZipFile(rt_zips[0], "r") as zf:
            csv_names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
            target_name = next((n for n in csv_names if "sepsis_realtime_hourly" in n.lower()), csv_names[0])
            with zf.open(target_name) as fp:
                df = pd.read_csv(fp)
        print(f"Loaded zipped real-time cohort: {rt_zips[0]} -> {len(df):,} rows, {df['patient_id'].nunique():,} patients")
        return df

    # Priority 3: Raw PhysioNet .psv files (deduplicate patient_id if both top-level and raw_psv/ contain a copy)
    psv_files = sorted(f for f in all_files if f.lower().endswith(".psv"))
    if psv_files:
        psv_by_pid = {}
        for path in psv_files:
            pid = os.path.splitext(os.path.basename(path))[0]
            in_raw_sub = "raw_psv" in path.replace("\\", "/").lower()
            if pid not in psv_by_pid or in_raw_sub:
                psv_by_pid[pid] = path
        frames = []
        for pid, path in sorted(psv_by_pid.items()):
            part = pd.read_csv(path, sep="|")
            part["patient_id"] = pid
            frames.append(part)
        df = pd.concat(frames, ignore_index=True)
        print(f"Loaded {len(psv_by_pid):,} unique raw PSV trajectories (from {len(psv_files):,} .psv files) -> {len(df):,} hourly rows")
        return df

    # Priority 4: Any .zip archive containing .psv trajectories or CSV
    any_zips = sorted(f for f in all_files if f.lower().endswith(".zip"))
    for zpath in any_zips:
        with zipfile.ZipFile(zpath, "r") as zf:
            z_psvs = [n for n in zf.namelist() if n.lower().endswith(".psv")]
            if z_psvs:
                psv_by_pid = {}
                for name in z_psvs:
                    pid = os.path.splitext(os.path.basename(name))[0]
                    if pid not in psv_by_pid or "raw_psv" in name.lower():
                        psv_by_pid[pid] = name
                frames = []
                for pid, name in sorted(psv_by_pid.items()):
                    with zf.open(name) as fp:
                        part = pd.read_csv(fp, sep="|")
                    part["patient_id"] = pid
                    frames.append(part)
                df = pd.concat(frames, ignore_index=True)
                print(f"Loaded {len(psv_by_pid):,} PSV trajectories from zip {zpath} -> {len(df):,} rows")
                return df

    # Priority 5: Fallback combined CSVs / legacy colleague CSVs
    all_csvs = sorted(f for f in all_files if f.lower().endswith(".csv"))
    combined = [
        f for f in all_csvs
        if "manifest" not in os.path.basename(f).lower()
        and "conversion" not in os.path.basename(f).lower()
        and not os.path.basename(f).startswith("P")
    ]
    if combined:
        df = pd.read_csv(combined[0])
        print(f"Loaded CSV fallback: {combined[0]} -> {len(df):,} rows, {df['patient_id'].nunique():,} patients")
        return df

    listed = {r: sorted(os.listdir(r))[:10] for r in roots}
    raise FileNotFoundError(
        f"No .csv, .zip, or .psv dataset files found under {roots} (top-level contents: {listed}). "
        "If your dataset appears in Kaggle's Input sidebar, click Session -> Restart Session (or paste its copied path into CUSTOM_INPUT_PATH in Cell 1) and run all cells from Cell 1."
    )

df_raw = load_realtime_icu_cohort(INPUT_ROOT)
df_raw = df_raw.sort_values(["patient_id", "ICULOS"]).reset_index(drop=True)

n_patients = df_raw["patient_id"].nunique()
map_med = abs(df_raw["MAP"].dropna().median())
IS_ZSCORED = bool(map_med < 15.0)
print(f"Patients: {n_patients:,} | Rows: {len(df_raw):,} | Scale: {'Z-SCORED (WARNING: attach sepsis_realtime_cohort.zip!)' if IS_ZSCORED else f'RAW PHYSICAL UNITS (median MAP={map_med:.1f} mmHg)'}")
if n_patients < 100 or IS_ZSCORED:
    print("ALERT: You appear to have attached the 24-patient z-scored medoids instead of sepsis_realtime_cohort.zip. Attach data/kaggle_export/sepsis_realtime_cohort.zip for high AUROC real-time forecasting!")

CORE_TELEMETRY = [
    "HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp", "FiO2", "pH", "BaseExcess",
    "BUN", "Creatinine", "Glucose", "WBC", "Platelets", "Lactate", "Bilirubin_total",
    "Hgb", "Hct", "Chloride", "Age", "HospAdmTime", "ICULOS"
]
audit = []
for col in CORE_TELEMETRY:
    if col in df_raw.columns:
        s = df_raw[col]
        audit.append({
            "Column": col,
            "Observed": int(s.notna().sum()),
            "RawMissingPct": round(float(s.isna().mean() * 100), 1),
            "Mean": round(float(s.mean()), 2) if s.notna().any() else np.nan,
            "Median": round(float(s.median()), 2) if s.notna().any() else np.nan,
            "SD": round(float(s.std()), 2) if s.notna().sum() > 1 else np.nan,
        })
display(pd.DataFrame(audit))


### Telemetry Schema & Arrival Audit Interpretation
Examine `RawMissingPct`, `Mean`, and `Median`:
- Bedside vitals (`HR`, `O2Sat`, `MAP`, `Resp`) arrive nearly every hour (`~10-16%` raw missingness), whereas labs (`Lactate`, `Bilirubin_total`, `Creatinine`, `WBC`) arrive asynchronously (`90-98%` raw hourly missingness between draws).
- Preserving raw `NaN` alongside causal forward-fill allows tree models (`HistGradientBoostingClassifier`) to treat **clinical test-ordering sparsity** as informative signal rather than corrupting distributions with global mean imputation.


## 3. Exploratory Cohort Analysis: Class Imbalance, Onset Timing & Arrival Dynamics
Analyzes patient stay lengths, sepsis-3 onset timing (`early admission` vs `late ICU-acquired`), and how raw telemetry varies inside the 6-hour forecast window.


In [ ]:
pt_stats = df_raw.groupby("patient_id").agg(
    n_hours=("ICULOS", "count"),
    has_sepsis=("SepsisLabel", "max"),
    pos_hours=("SepsisLabel", "sum"),
).reset_index()

onset_rows = df_raw[df_raw["SepsisLabel"] == 1].groupby("patient_id")["ICULOS"].min().rename("t_first_label1")
pt_stats = pt_stats.merge(onset_rows, on="patient_id", how="left")

pos_row_rate = df_raw["SepsisLabel"].mean() * 100
pos_pt_rate = pt_stats["has_sepsis"].mean() * 100
counts = df_raw["SepsisLabel"].value_counts().reindex([0, 1], fill_value=0)
imbalance_ratio = counts.get(0, 0) / max(1, counts.get(1, 1))
early_admit = int((pt_stats["t_first_label1"] <= 2).sum())
late_icu = int((pt_stats["t_first_label1"] > 20).sum())

print(f"Total Patients: {len(pt_stats):,} | Sepsis+: {int(pt_stats['has_sepsis'].sum()):,} ({pos_pt_rate:.2f}%)")
print(f"Hourly Rows: {len(df_raw):,} | Positive 6h-Forecast Rows: {int(df_raw['SepsisLabel'].sum()):,} ({pos_row_rate:.2f}% | {imbalance_ratio:.1f}:1 imbalance)")
print(f"Imbalance Strategy: Train uses sqrt inverse-prevalence pos_weight + Pairwise AUROC loss; Val/Test preserve real {pos_row_rate:.2f}% ICU incidence for honest calibration & PPV.")
print(f"Sepsis Timing: Early Admission (ICULOS<=2) = {early_admit} | Late ICU-Acquired (ICULOS>20) = {late_icu}")

fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.6))
x_imb = np.arange(2)
w_imb = 0.36
neg_vals = [100.0 - pos_row_rate, 100.0 - pos_pt_rate]
pos_vals = [pos_row_rate, pos_pt_rate]
b_neg = axes[0].bar(x_imb - w_imb / 2, neg_vals, width=w_imb, color="#008775", label="Control / Non-Sepsis (0)")
b_pos = axes[0].bar(x_imb + w_imb / 2, pos_vals, width=w_imb, color="#b91c1c", label="Sepsis+ / 6h Window (1)")
axes[0].set_ylim(0, 118)
axes[0].set_xticks(x_imb)
axes[0].set_xticklabels([f"Hourly Ticks\n({imbalance_ratio:.0f}:1)", f"ICU Patients\n({(100-pos_pt_rate)/max(1e-3,pos_pt_rate):.0f}:1)"])
axes[0].set_ylabel("Cohort Share (%)")
axes[0].set_title("Real ICU Class Imbalance (Hourly vs Patient)")
axes[0].legend(fontsize=8, loc="upper right")
hr_cnts = [int(counts.get(0, 0)), int(pt_stats["has_sepsis"].eq(0).sum())]
pt_cnts = [int(counts.get(1, 0)), int(pt_stats["has_sepsis"].eq(1).sum())]
for rect, cnt, pct in zip(b_neg, hr_cnts, neg_vals):
    axes[0].text(rect.get_x() + rect.get_width() / 2, pct + 2.0, f"{pct:.1f}%\n({cnt:,})", ha="center", va="bottom", fontsize=8)
for rect, cnt, pct in zip(b_pos, pt_cnts, pos_vals):
    axes[0].text(rect.get_x() + rect.get_width() / 2, pct + 2.0, f"{pct:.2f}%\n({cnt:,})", ha="center", va="bottom", fontsize=8, fontweight="bold", color="#991b1b")

axes[1].hist(pt_stats["t_first_label1"].dropna(), bins=25, color="#0284c7", edgecolor="#0f172a", alpha=0.85)
axes[1].axvline(20, color="#b91c1c", linestyle="--", label="Late ICU Threshold (20h)")
axes[1].set_title("Sepsis-3 Forecast Window Start Hour")
axes[1].set_xlabel("ICULOS at First SepsisLabel=1 (hours)")
axes[1].set_ylabel("Sepsis+ Patients")
axes[1].legend()

# Plot vital separation (raw observations) between Class 0 and Class 1
avail_raw = [c for c in ["HR", "MAP", "Resp", "Temp", "Lactate", "BUN", "Creatinine", "WBC", "Platelets", "ICULOS"] if c in df_raw.columns]
uni = []
for col in avail_raw:
    sub = df_raw[[col, "SepsisLabel"]].dropna()
    if sub["SepsisLabel"].nunique() == 2 and len(sub) > 50:
        auc = roc_auc_score(sub["SepsisLabel"], sub[col])
        uni.append({"Feature": col, "DirAUROC": max(auc, 1.0 - auc), "PosHigher": auc >= 0.5})
uni_df = pd.DataFrame(uni).sort_values("DirAUROC", ascending=True)
axes[2].barh(uni_df["Feature"], uni_df["DirAUROC"], color=["#b91c1c" if h else "#008775" for h in uni_df["PosHigher"]])
axes[2].axvline(0.50, color="#334155", linestyle="--")
axes[2].set_title("Raw Single-Variable Separability (Red=+ Higher)")
axes[2].set_xlabel("Directional AUROC")
plt.tight_layout()
plt.show()


### EDA Takeaways for Class Imbalance & Feature Engineering
- **Real ICU Epidemiology (`~43:1` Hourly vs `~9%` Patient-Level):** Only `~2.25%` of total ICU patient-hours fall inside a 6-hour pre-sepsis window. We intentionally **never oversample or SMOTE `Val`/`Test`** (doing so fabricates PPV and corrupts bedside probability calibration). Instead, **`Train`** addresses imbalance natively with capped inverse-prevalence `pos_weight` (`sqrt(n_neg/n_pos)`) and **Pairwise AUROC Ranking Loss**.
- Because a large share of sepsis cases emerge **late in the ICU stay (`ICULOS > 20h`)**, absolute cutoffs alone miss patients who deteriorate relative to their own resting baseline (e.g., an older hypertensive patient dropping from `MAP 105` down to `75 mmHg`).
- Adding **patient-personalized causal expanding deviations** (`x(t) - mean(x[0..t])`) alongside multi-window vital trends (`6h / 12h`) and clinical subscores directly addresses both early-admission shock and gradual late-ICU decompensation.


## 4. Strict Causal Real-Time Streaming Feature Engineering (Zero Leakage, Browser-Parity)
Generates the exact **79 causal features** mirrored identically in `app/js/predictiveModel.js` and `scripts/train_predictive_model.py`:
- At hour `t`, calculations see **strictly `<= t`** (no centered rolling windows, no future normalization).
- Returns **both** a causal LOCF matrix (for scaling & Linear/GBDT export) and a **missingness-preserved native-NaN matrix** for `HistGradientBoostingClassifier`.


In [ ]:
FEATURES = [
    "HR", "O2Sat", "Temp", "MAP", "Resp", "SBP", "DBP",
    "BUN", "Creatinine", "Glucose", "WBC", "Platelets", "Lactate",
    "Bilirubin_total", "FiO2", "pH", "BaseExcess", "Hgb", "Hct", "Chloride",
    "Age", "HospAdmTime", "ICULOS",
    "ShockIndex", "PulsePressure", "ROX_proxy", "Shock_x_Resp", "MAP_to_Age", "BUN_Creat_ratio",
    "Temp_deviation", "Temp_high_hinge", "Temp_low_hinge",
    "WBC_high_hinge", "WBC_low_hinge", "Lactate_hinge", "Platelet_low_hinge", "Creat_high_hinge",
    "Hrs_since_Lactate", "Hrs_since_WBC", "Hrs_since_Creatinine", "Lab_update_events",
    "HR_diff_base", "MAP_diff_base", "Resp_diff_base", "Temp_diff_base", "O2Sat_diff_base", "SI_diff_base",
    "HR_delta_1h", "MAP_delta_1h",
    "HR_mean_6h", "HR_max_6h", "HR_std_6h", "HR_delta_6h", "HR_drift_12h", "HR_tachy_frac_6h",
    "MAP_min_6h", "MAP_mean_6h", "MAP_std_6h", "MAP_delta_6h", "MAP_drift_12h", "MAP_hypo_frac_6h",
    "O2Sat_min_6h", "O2Sat_mean_6h",
    "Resp_max_6h", "Resp_mean_6h", "Resp_delta_6h",
    "Temp_max_6h", "Temp_min_6h", "SI_max_6h", "SI_mean_6h",
    "Creatinine_delta", "Platelets_delta", "WBC_delta", "BUN_delta", "Lactate_delta",
    "qSOFA_partial", "SIRS_count", "SOFA_partial", "NEWS2_proxy",
]

def _news2(rp, o2, sbp, mp, hr, tp):
    pts = 0.0
    if rp is not None:
        if rp <= 8 or rp >= 25: pts += 3.0
        elif rp >= 21: pts += 2.0
        elif rp <= 11: pts += 1.0
    if o2 is not None:
        if o2 <= 91: pts += 3.0
        elif o2 <= 93: pts += 2.0
        elif o2 <= 95: pts += 1.0
    if sbp is not None:
        if sbp <= 90 or sbp >= 220: pts += 3.0
        elif sbp <= 100: pts += 2.0
        elif sbp <= 110: pts += 1.0
    elif mp is not None:
        if mp <= 60: pts += 3.0
        elif mp <= 65: pts += 2.0
        elif mp <= 70: pts += 1.0
    if hr is not None:
        if hr <= 40 or hr >= 131: pts += 3.0
        elif hr >= 111: pts += 2.0
        elif hr <= 50 or hr >= 91: pts += 1.0
    if tp is not None:
        if tp <= 35.0: pts += 3.0
        elif tp >= 39.1: pts += 2.0
        elif tp <= 36.0 or tp >= 38.1: pts += 1.0
    return pts

def _sofa_p(mp, o2, cr, plt_v, bili, fio2):
    cv = 1 if (mp is not None and mp < 70) else 0
    ren = 0 if cr is None else (4 if cr >= 5 else (3 if cr >= 3.5 else (2 if cr >= 2 else (1 if cr >= 1.2 else 0))))
    coag = 0 if plt_v is None else (4 if plt_v < 20 else (3 if plt_v < 50 else (2 if plt_v < 100 else (1 if plt_v < 150 else 0))))
    liv = 0 if bili is None else (4 if bili >= 12 else (3 if bili >= 6 else (2 if bili >= 2 else (1 if bili >= 1.2 else 0))))
    resp = 0
    if o2 is not None:
        f = fio2 if (fio2 is not None and fio2 > 0) else 0.21
        if f > 1.0: f /= 100.0
        rat = o2 / max(0.21, f)
        resp = 4 if rat < 100 else (3 if rat < 200 else (2 if rat < 300 else (1 if rat < 400 else 0)))
    return float(cv + ren + coag + liv + resp)

RAW_COLS = [
    "HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp", "FiO2",
    "BUN", "Creatinine", "Glucose", "WBC", "Platelets", "Lactate",
    "Bilirubin_total", "pH", "BaseExcess", "Hgb", "Hct", "Chloride",
    "Age", "HospAdmTime", "ICULOS",
]

def build_causal_streaming_features(df_in):
    t0 = time.time()
    n_rows = len(df_in)
    n_feat = len(FEATURES)
    X_locf = np.full((n_rows, n_feat), np.nan, dtype=np.float32)

    col_arrays = {c: (df_in[c].to_numpy(dtype=np.float64) if c in df_in.columns else np.full(n_rows, np.nan)) for c in RAW_COLS}
    pids = df_in["patient_id"].to_numpy()
    # Find patient boundary indices in O(n)
    boundaries = np.flatnonzero(pids[1:] != pids[:-1]) + 1
    starts = np.concatenate(([0], boundaries))
    ends = np.concatenate((boundaries, [n_rows]))

    lab_track = ("Lactate", "WBC", "Creatinine", "Platelets", "Bilirubin_total")
    base_track = ("Creatinine", "WBC", "Platelets", "BUN", "Lactate")

    for s_idx, e_idx in zip(starts, ends):
        last = {c: None for c in RAW_COLS}
        base_lab = {}
        hrs_since = {c: 24.0 for c in lab_track}
        lab_events = 0.0
        hist = {"HR": [], "MAP": [], "O2Sat": [], "Resp": [], "Temp": [], "SI": []}
        exp_sum = {k: 0.0 for k in hist}
        exp_cnt = {k: 0 for k in hist}

        for row_i in range(s_idx, e_idx):
            any_upd = False
            for lc in lab_track:
                v = col_arrays[lc][row_i]
                if not math.isnan(v):
                    if last[lc] is None or abs(v - last[lc]) > 1e-6:
                        hrs_since[lc] = 0.0
                        any_upd = True
                    else:
                        hrs_since[lc] = min(48.0, hrs_since[lc] + 1.0)
                else:
                    hrs_since[lc] = min(48.0, hrs_since[lc] + 1.0)
            if any_upd:
                lab_events += 1.0

            for c in RAW_COLS:
                v = col_arrays[c][row_i]
                if not math.isnan(v):
                    fv = float(v)
                    last[c] = fv
                    if c in base_track and c not in base_lab:
                        base_lab[c] = fv
            if last["ICULOS"] is None:
                last["ICULOS"] = float(row_i - s_idx + 1)

            hr = last["HR"]; mp = last["MAP"]; o2 = last["O2Sat"]; rp = last["Resp"]; tp = last["Temp"]
            sbp = last["SBP"]; dbp = last["DBP"]; age = last["Age"]
            si = (hr / mp) if (hr is not None and mp is not None and mp > 10.0) else None
            pp = (sbp - dbp) if (sbp is not None and dbp is not None and sbp > dbp) else None
            fio2 = last["FiO2"]
            fio2_norm = (fio2 / 100.0 if fio2 > 1.0 else fio2) if (fio2 is not None and fio2 > 0) else 0.21
            rox = ((o2 / max(0.21, fio2_norm)) / max(4.0, rp)) if (o2 is not None and rp is not None) else None

            for key, val in (("HR", hr), ("MAP", mp), ("O2Sat", o2), ("Resp", rp), ("Temp", tp), ("SI", si)):
                if val is not None:
                    hist[key].append(val)
                    exp_sum[key] += val
                    exp_cnt[key] += 1

            w_hr = hist["HR"][-6:]; w_mp = hist["MAP"][-6:]; w_o2 = hist["O2Sat"][-6:]
            w_rp = hist["Resp"][-6:]; w_tp = hist["Temp"][-6:]; w_si = hist["SI"][-6:]
            w_hr12 = hist["HR"][-12:]; w_mp12 = hist["MAP"][-12:]
            hr_m6 = sum(w_hr) / len(w_hr) if w_hr else np.nan
            mp_m6 = sum(w_mp) / len(w_mp) if w_mp else np.nan
            hr_std6 = math.sqrt(sum((x - hr_m6) ** 2 for x in w_hr) / len(w_hr)) if len(w_hr) >= 2 else 0.0
            mp_std6 = math.sqrt(sum((x - mp_m6) ** 2 for x in w_mp) / len(w_mp)) if len(w_mp) >= 2 else 0.0

            qsofa = (1.0 if (rp is not None and rp >= 22) else 0.0) + (1.0 if ((sbp is not None and sbp <= 100) or (mp is not None and mp <= 65)) else 0.0)
            sirs = (
                (1.0 if (tp is not None and (tp > 38.0 or tp < 36.0)) else 0.0)
                + (1.0 if (hr is not None and hr > 90) else 0.0)
                + (1.0 if (rp is not None and rp > 20) else 0.0)
                + (1.0 if (last["WBC"] is not None and (last["WBC"] > 12.0 or last["WBC"] < 4.0)) else 0.0)
            )
            def nn(v): return np.nan if v is None else v
            def ed(k, cur): return (cur - exp_sum[k] / exp_cnt[k]) if (cur is not None and exp_cnt[k] >= 2) else 0.0

            X_locf[row_i] = [
                nn(hr), nn(o2), nn(tp), nn(mp), nn(rp), nn(sbp), nn(dbp),
                nn(last["BUN"]), nn(last["Creatinine"]), nn(last["Glucose"]), nn(last["WBC"]), nn(last["Platelets"]), nn(last["Lactate"]),
                nn(last["Bilirubin_total"]), nn(last["FiO2"]), nn(last["pH"]), nn(last["BaseExcess"]), nn(last["Hgb"]), nn(last["Hct"]), nn(last["Chloride"]),
                nn(age), nn(last["HospAdmTime"]), nn(last["ICULOS"]),
                nn(si), nn(pp), nn(rox),
                (si * rp) if (si is not None and rp is not None) else np.nan,
                (mp / max(20.0, age)) if (mp is not None and age is not None) else np.nan,
                (last["BUN"] / max(0.2, last["Creatinine"])) if (last["BUN"] is not None and last["Creatinine"] is not None) else np.nan,
                abs(tp - 37.0) if tp is not None else np.nan,
                max(0.0, tp - 38.0) if tp is not None else 0.0,
                max(0.0, 36.0 - tp) if tp is not None else 0.0,
                max(0.0, last["WBC"] - 12.0) if last["WBC"] is not None else 0.0,
                max(0.0, 4.0 - last["WBC"]) if last["WBC"] is not None else 0.0,
                max(0.0, last["Lactate"] - 2.0) if last["Lactate"] is not None else 0.0,
                max(0.0, 150.0 - last["Platelets"]) if last["Platelets"] is not None else 0.0,
                max(0.0, last["Creatinine"] - 1.2) if last["Creatinine"] is not None else 0.0,
                hrs_since["Lactate"], hrs_since["WBC"], hrs_since["Creatinine"], lab_events,
                ed("HR", hr), ed("MAP", mp), ed("Resp", rp), ed("Temp", tp), ed("O2Sat", o2), ed("SI", si),
                (w_hr[-1] - w_hr[-2]) if len(w_hr) >= 2 else 0.0,
                (w_mp[-1] - w_mp[-2]) if len(w_mp) >= 2 else 0.0,
                hr_m6, max(w_hr) if w_hr else np.nan, hr_std6,
                (w_hr[-1] - w_hr[0]) if len(w_hr) >= 2 else 0.0,
                (w_hr[-1] - sum(w_hr12) / len(w_hr12)) if len(w_hr12) >= 2 else 0.0,
                sum(1 for x in w_hr if x > 100.0) / len(w_hr) if w_hr else 0.0,
                min(w_mp) if w_mp else np.nan, mp_m6, mp_std6,
                (w_mp[-1] - w_mp[0]) if len(w_mp) >= 2 else 0.0,
                (w_mp[-1] - sum(w_mp12) / len(w_mp12)) if len(w_mp12) >= 2 else 0.0,
                sum(1 for x in w_mp if x < 65.0) / len(w_mp) if w_mp else 0.0,
                min(w_o2) if w_o2 else np.nan, sum(w_o2) / len(w_o2) if w_o2 else np.nan,
                max(w_rp) if w_rp else np.nan, sum(w_rp) / len(w_rp) if w_rp else np.nan,
                (w_rp[-1] - w_rp[0]) if len(w_rp) >= 2 else 0.0,
                max(w_tp) if w_tp else np.nan, min(w_tp) if w_tp else np.nan,
                max(w_si) if w_si else np.nan, sum(w_si) / len(w_si) if w_si else np.nan,
                (last["Creatinine"] - base_lab["Creatinine"]) if ("Creatinine" in base_lab and last["Creatinine"] is not None) else 0.0,
                (last["Platelets"] - base_lab["Platelets"]) if ("Platelets" in base_lab and last["Platelets"] is not None) else 0.0,
                (last["WBC"] - base_lab["WBC"]) if ("WBC" in base_lab and last["WBC"] is not None) else 0.0,
                (last["BUN"] - base_lab["BUN"]) if ("BUN" in base_lab and last["BUN"] is not None) else 0.0,
                (last["Lactate"] - base_lab["Lactate"]) if ("Lactate" in base_lab and last["Lactate"] is not None) else 0.0,
                qsofa, sirs,
                _sofa_p(mp, o2, last["Creatinine"], last["Platelets"], last["Bilirubin_total"], last["FiO2"]),
                _news2(rp, o2, sbp, mp, hr, tp),
            ]
    print(f"Extracted {n_feat} causal streaming features across {len(starts):,} patients in {time.time() - t0:.2f}s")
    return X_locf

X_all_nan = build_causal_streaming_features(df_raw)
y_all = df_raw["SepsisLabel"].to_numpy(dtype=np.int32)
groups_all = df_raw["patient_id"].to_numpy()

# Stratified Patient-Level 70 / 15 / 15 Split (Zero Patient Overlap)
pt_y = pt_stats.set_index("patient_id")["has_sepsis"]
dev_pts, test_pts, dev_y, _ = train_test_split(pt_y.index.to_numpy(), pt_y.to_numpy(), test_size=0.15, stratify=pt_y.to_numpy(), random_state=42)
train_pts, val_pts = train_test_split(dev_pts, test_size=(0.15 / 0.85), stratify=dev_y, random_state=42)

tr_mask = df_raw["patient_id"].isin(train_pts).to_numpy()
va_mask = df_raw["patient_id"].isin(val_pts).to_numpy()
te_mask = df_raw["patient_id"].isin(test_pts).to_numpy()

assert len(set(train_pts) & set(val_pts)) == 0
assert len(set(train_pts) & set(test_pts)) == 0
assert len(set(val_pts) & set(test_pts)) == 0

# Fit Standardization Exclusively on Train Split
train_means = np.nan_to_num(np.nanmean(X_all_nan[tr_mask], axis=0), nan=0.0)
train_sds = np.nan_to_num(np.nanstd(X_all_nan[tr_mask], axis=0), nan=1.0)
train_sds = np.maximum(train_sds, 1e-4)

def to_native_nan_z(X_sub):
    return np.clip((X_sub - train_means) / train_sds, -5.0, 5.0)

def to_imputed_z(X_sub):
    Z = (X_sub - train_means) / train_sds
    Z = np.where(np.isnan(Z), 0.0, Z)
    return np.clip(Z, -5.0, 5.0)

X_tr_nan = to_native_nan_z(X_all_nan[tr_mask])
X_va_nan = to_native_nan_z(X_all_nan[va_mask])
X_te_nan = to_native_nan_z(X_all_nan[te_mask])

X_tr_imp = to_imputed_z(X_all_nan[tr_mask])
X_va_imp = to_imputed_z(X_all_nan[va_mask])
X_te_imp = to_imputed_z(X_all_nan[te_mask])

y_tr, y_va, y_te = y_all[tr_mask], y_all[va_mask], y_all[te_mask]
g_tr, g_va, g_te = groups_all[tr_mask], groups_all[va_mask], groups_all[te_mask]

for name, m_mask in [("Train", tr_mask), ("Val  ", va_mask), ("Test ", te_mask)]:
    sub_df = df_raw.loc[m_mask]
    n_p = sub_df["patient_id"].nunique()
    pos_p = int(sub_df.groupby("patient_id")["SepsisLabel"].max().sum())
    print(f"{name}: {n_p:,} pts ({pos_p}+ / {n_p - pos_p}-) | {len(sub_df):,} rows ({int(sub_df['SepsisLabel'].sum()):,} pos)")


### Split & Leakage Verification
Look at the split counts above: `Train`, `Val`, and `Test` maintain identical sepsis prevalence at both the patient level (`~9.2%`) and hourly level (`~2.25%`) with **zero patient leakage**, and `train_means` / `train_sds` are fit strictly on `Train`.


## 5. Comprehensive Monitoring Helper Functions & Train-vs-Test Feature Drift Audit
Before training complex models, we define the full production clinical monitoring suite (PhysioNet Challenge 2019 Normalized Utility, Calibration `Brier`/`ECE`, Alarm Burden `FP/100h`, Early Lead Time) and verify **Train-vs-Test Feature Drift (`PSI`)**.


In [ ]:
def physionet_utility_single(ys, preds):
    has_sep = bool(np.any(ys == 1))
    t_sep = (int(np.argmax(ys == 1)) + 6) if has_sep else None
    u = u_opt = u_null = 0.0
    for t, pr in enumerate(preds):
        if has_sep:
            dt = t - t_sep
            if -12 <= dt <= -6:
                wt, wn_ = (dt + 12) / 6.0, 0.0
            elif -6 < dt <= 0:
                wt, wn_ = 1.0, -2.0 * (dt + 6) / 6.0
            elif 0 < dt <= 3:
                wt, wn_ = 1.0 - dt / 3.0, -2.0
            elif dt < -12:
                wt, wn_ = -0.05, 0.0
            else:
                wt, wn_ = 0.0, 0.0
        else:
            wt, wn_ = -0.05, 0.0
        u += wt if pr == 1 else wn_
        u_opt += max(wt, wn_)
        u_null += wn_
    return u, u_opt, u_null

def compute_ece(y_true, probs, n_bins=10):
    ece = 0.0
    n = len(y_true)
    for b in range(n_bins):
        lo, hi = b / n_bins, (b + 1) / n_bins
        mask = (probs >= lo) & (probs < hi) if b < n_bins - 1 else (probs >= lo) & (probs <= hi)
        if np.any(mask):
            ece += (np.sum(mask) / n) * abs(float(np.mean(y_true[mask])) - float(np.mean(probs[mask])))
    return float(ece)

def persist_2of3_seq(ps, th):
    hits = (np.asarray(ps) >= th).astype(int)
    s = hits.copy()
    if len(hits) >= 2:
        s[1:] += hits[:-1]
    if len(hits) >= 3:
        s[2:] += hits[:-2]
    return (s >= 2).astype(int)

def evaluate_monitoring_suite(y_true, probs, patient_ids, threshold=None, watchlist_threshold=None, max_fp_per_100h=5.0):
    y_true = np.asarray(y_true, dtype=int)
    probs = np.asarray(probs, dtype=np.float64)
    auroc = roc_auc_score(y_true, probs)
    auprc = average_precision_score(y_true, probs)
    brier = brier_score_loss(y_true, probs)
    ece = compute_ece(y_true, probs)

    df_eval = pd.DataFrame({"patient_id": patient_ids, "y": y_true, "p": probs})
    grouped = [(pid, sub["y"].to_numpy(), sub["p"].to_numpy()) for pid, sub in df_eval.groupby("patient_id", sort=False)]

    # 1. Watchlist threshold: maximizes normalized PhysioNet utility (passive nurse screen)
    if watchlist_threshold is None:
        cands_w = np.quantile(probs, np.linspace(0.70, 0.99, 60))
        best_w, best_u = 0.03, -1e9
        for cand in np.unique(np.round(cands_w, 4)):
            us = uo_s = un_s = 0.0
            for _, ys, ps in grouped:
                u, uo, un = physionet_utility_single(ys, (ps >= cand).astype(int))
                us += u; uo_s += uo; un_s += un
            unorm = (us - un_s) / max(1e-9, uo_s - un_s)
            if unorm > best_u:
                best_u, best_w = unorm, float(cand)
        watchlist_threshold = best_w

    # 2. Primary Critical Actionable Alert threshold: lowest threshold meeting alarm budget (FP <= max_fp_per_100h)
    if threshold is None:
        cands_c = np.unique(np.round(np.quantile(probs, np.linspace(0.72, 0.996, 260)), 4))
        threshold = float(cands_c[-1])
        for cand in cands_c:
            fp100_c = 100.0 * float(np.sum((y_true == 0) & (probs >= cand))) / max(1, len(y_true))
            if fp100_c <= max_fp_per_100h:
                threshold = float(cand)
                break

    us = uo_s = un_s = 0.0
    us_w = 0.0
    pt_y, pt_p, pt_pred_2of3 = [], [], []
    leads_24h, leads_12h, watch_leads_24h = [], [], []
    for _, ys, ps in grouped:
        u, uo, un = physionet_utility_single(ys, (ps >= threshold).astype(int))
        uw, _, _ = physionet_utility_single(ys, (ps >= watchlist_threshold).astype(int))
        us += u; us_w += uw; uo_s += uo; un_s += un
        has_s = int(np.max(ys))
        pt_y.append(has_s)
        pt_p.append(float(np.max(ps)))
        pt_pred_2of3.append(int(np.any(persist_2of3_seq(ps, threshold))))
        if has_s == 1:
            t_onset = int(np.argmax(ys == 1)) + 6
            w24 = ps[max(0, t_onset - 24) : min(len(ps), t_onset + 4)]
            w12 = ps[max(0, t_onset - 12) : min(len(ps), t_onset + 4)]
            idx24 = np.flatnonzero(w24 >= threshold)
            idx12 = np.flatnonzero(w12 >= threshold)
            idx24_w = np.flatnonzero(w24 >= watchlist_threshold)
            if len(idx24) > 0:
                leads_24h.append(t_onset - (max(0, t_onset - 24) + int(idx24[0])))
            if len(idx12) > 0:
                leads_12h.append(t_onset - (max(0, t_onset - 12) + int(idx12[0])))
            if len(idx24_w) > 0:
                watch_leads_24h.append(t_onset - (max(0, t_onset - 24) + int(idx24_w[0])))

    pt_y_arr = np.asarray(pt_y, dtype=int)
    pt_p_arr = np.asarray(pt_p, dtype=np.float64)
    pt_2of3_arr = np.asarray(pt_pred_2of3, dtype=int)

    util_norm = (us - un_s) / max(1e-9, uo_s - un_s)
    util_watch = (us_w - un_s) / max(1e-9, uo_s - un_s)
    pt_auroc = roc_auc_score(pt_y_arr, pt_p_arr)
    pt_auprc = average_precision_score(pt_y_arr, pt_p_arr)

    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    pt_tn, pt_fp, pt_fn, pt_tp = confusion_matrix(pt_y_arr, pt_2of3_arr, labels=[0, 1]).ravel()

    sens = tp / max(1, tp + fn)
    spec = tn / max(1, tn + fp)
    ppv = tp / max(1, tp + fp)
    npv = tn / max(1, tn + fn)
    f1 = f1_score(y_true, preds, zero_division=0)
    return {
        "Hourly AUROC": round(float(auroc), 4),
        "Patient AUROC": round(float(pt_auroc), 4),
        "AUPRC": round(float(auprc), 4),
        "Pt_AUPRC": round(float(pt_auprc), 4),
        "Utility Score": round(float(util_watch), 4),
        "Crit Utility": round(float(util_norm), 4),
        "Brier": round(float(brier), 4),
        "ECE": round(float(ece), 4),
        "Watchlist_Th": round(float(watchlist_threshold), 4),
        "Threshold": round(float(threshold), 4),
        "Sensitivity": round(float(sens), 4),
        "Specificity": round(float(spec), 4),
        "PPV": round(float(ppv), 4),
        "NPV": round(float(npv), 4),
        "F1": round(float(f1), 4),
        "FP_per_100h": round(float(100.0 * fp / max(1, len(y_true))), 2),
        "Pt_2of3_Sens": round(float(pt_tp / max(1, pt_tp + pt_fn)), 4),
        "Pt_2of3_Spec": round(float(pt_tn / max(1, pt_tn + pt_fp)), 4),
        "Pt_2of3_PPV": round(float(pt_tp / max(1, pt_tp + pt_fp)), 4),
        "Watch_24h": f"{len(watch_leads_24h)}/{int(np.sum(pt_y_arr))}",
        "Caught_24h": f"{len(leads_24h)}/{int(np.sum(pt_y_arr))}",
        "Caught_12h": f"{len(leads_12h)}/{int(np.sum(pt_y_arr))}",
        "Median_Lead_h": round(float(np.median(leads_24h)), 1) if leads_24h else None,
        "probs": probs,
        "pt_y": pt_y_arr,
        "pt_p": pt_p_arr,
        "pt_2of3_preds": pt_2of3_arr,
        "threshold_raw": threshold,
        "watchlist_th_raw": watchlist_threshold,
    }

# Monitor Feature Distribution Drift (Train vs Test PSI)
drift_rows = []
for col in ["HR", "MAP", "Temp", "Resp", "O2Sat", "BUN", "Creatinine", "WBC", "Platelets", "Lactate", "ICULOS", "NEWS2_proxy"]:
    idx_f = FEATURES.index(col)
    tr_v = X_all_nan[tr_mask, idx_f]
    te_v = X_all_nan[te_mask, idx_f]
    tr_valid = tr_v[~np.isnan(tr_v)]
    te_valid = te_v[~np.isnan(te_v)]
    if len(tr_valid) > 20 and len(te_valid) > 20:
        edges = np.quantile(tr_valid, [0.2, 0.4, 0.6, 0.8])
        b1 = np.bincount(np.searchsorted(edges, tr_valid), minlength=5) / len(tr_valid)
        b2 = np.bincount(np.searchsorted(edges, te_valid), minlength=5) / len(te_valid)
        b1, b2 = np.clip(b1, 1e-4, 1.0), np.clip(b2, 1e-4, 1.0)
        psi = float(np.sum((b2 - b1) * np.log(b2 / b1)))
        smd = float((np.mean(te_valid) - np.mean(tr_valid)) / max(1e-4, np.std(tr_valid)))
        drift_rows.append({"Feature": col, "PSI": round(psi, 4), "SMD": round(smd, 4), "TrainMissPct": round(float(np.mean(np.isnan(tr_v)) * 100), 1), "TestMissPct": round(float(np.mean(np.isnan(te_v)) * 100), 1)})

drift_df = pd.DataFrame(drift_rows).sort_values("PSI", ascending=False)
display(drift_df)


### Drift Monitoring Interpretation
A Population Stability Index (**PSI < 0.10**) confirms covariates remain stationary between `Train` and `Held-Out Test`, whereas `TrainMissPct` vs `TestMissPct` tracks lab ordering consistency before model training.


## 6. PLE Tabular Representation & 5-Fold PyTorch `PLE-ResNet` Training
Naive `NaN -> 0.0` mean imputation feeds normal-mean lab values (e.g., `Lactate = 2.47`) into neural networks whenever a test has not been ordered yet. To eliminate that distortion while unlocking deep tabular representation power:
- We build **Piecewise-Linear Quantile Encodings (`PLE`, fit strictly on non-missing `Train` observations)** across 28 core clinical predictors + `signed-log1p(z)` + `20` explicit raw vital/lab missingness masks (`~262D` vector).
- We compare five models on identical patient splits + Validation-fit **1D Platt Sigmoid Calibration**:
  1. **NEWS2/qSOFA Baseline:** Linear model over deterministic bedside subscores.
  2. **L2 Logistic Regression:** Regularized linear baseline (`C` tuned on Validation).
  3. **`HistGradientBoostingClassifier` (Reference):** Tree-based native-NaN benchmark.
  4. **`SwiGLUNet` (Neural Comparison):** Gated SwiGLU residual MLP on the exact `PLE` encoding.
  5. **`5-Fold StratifiedGroupKFold PLE-ResNet` (Primary Deployed Neural Network, `hid=64`):** `LayerNorm + SiLU + 2 Residual Blocks + 0.5*Skip` trained with **`0.55 * Weighted BCE + 0.45 * Pairwise AUROC Ranking Loss`** and patient-grouped 5-fold ensembling.


In [ ]:
def _logit(p):
    p = np.clip(p, 1e-6, 1.0 - 1e-6)
    return np.log(p / (1.0 - p))

def _sig(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -20.0, 20.0)))

# 1. Clinical Baseline & L2 Logistic Regression Reference
clin_idx = [FEATURES.index(f) for f in ["qSOFA_partial", "SIRS_count", "SOFA_partial", "NEWS2_proxy"]]
m_clin = LogisticRegression(C=0.5, class_weight="balanced", max_iter=500).fit(X_tr_imp[:, clin_idx], y_tr)

best_c, best_lr_val, m_lr = 0.05, -1.0, None
for c_val in [0.005, 0.02, 0.05, 0.2, 1.0]:
    cand = LogisticRegression(C=c_val, class_weight="balanced", max_iter=800, random_state=42).fit(X_tr_imp, y_tr)
    v_auc = roc_auc_score(y_va, cand.predict_proba(X_va_imp)[:, 1])
    if v_auc > best_lr_val:
        best_lr_val, best_c, m_lr = v_auc, c_val, cand
print(f"Best L2 LogReg (C={best_c}): Val AUROC={best_lr_val:.4f}")

# 2. HistGradientBoostingClassifier (Native-NaN Reference Benchmark)
m_hgb = HistGradientBoostingClassifier(
    max_iter=280, learning_rate=0.04, max_leaf_nodes=23, min_samples_leaf=80,
    l2_regularization=10.0, class_weight="balanced", early_stopping=True,
    validation_fraction=0.1, random_state=42
).fit(X_tr_nan, y_tr)
print(f"HistGB (Reference): Val AUROC={roc_auc_score(y_va, m_hgb.predict_proba(X_va_nan)[:, 1]):.4f}")

# 3. Leak-Free Piecewise-Linear Encoding (PLE) + Signed-Log1p + Missingness Masks (fit strictly on Train!)
PLE_COLS = [
    "HR", "O2Sat", "Temp", "MAP", "Resp", "SBP",
    "BUN", "Creatinine", "Glucose", "WBC", "Platelets", "Lactate", "Bilirubin_total",
    "FiO2", "pH", "Hgb", "Age", "HospAdmTime", "ICULOS",
    "ShockIndex", "ROX_proxy", "Hrs_since_Lactate", "Hrs_since_WBC", "Lab_update_events",
    "HR_diff_base", "MAP_diff_base", "Resp_diff_base", "Temp_diff_base",
]

def fit_ple_specs(X_train_nan, n_bins=6):
    specs = []
    qs = np.linspace(0.0, 1.0, n_bins + 1)
    for col in PLE_COLS:
        j = FEATURES.index(col)
        valid = X_train_nan[:, j]
        valid = valid[~np.isnan(valid)]
        if len(valid) < 20:
            continue
        raw_e = [round(float(x), 5) for x in np.quantile(valid, qs)]
        edges = [raw_e[0]]
        for e in raw_e[1:]:
            if e - edges[-1] > 1e-4:
                edges.append(e)
        if len(edges) >= 2:
            specs.append({"f": int(j), "name": col, "edges": edges})
    return specs

def encode_ple_matrix(X_sub_nan, specs):
    n_rows, n_feat = X_sub_nan.shape
    z_imp = np.nan_to_num(X_sub_nan, nan=0.0).astype(np.float32)
    slog = np.sign(z_imp) * np.log1p(np.abs(z_imp))
    miss20 = np.isnan(X_sub_nan[:, :20]).astype(np.float32)
    ple_cols = []
    for sp in specs:
        j = sp["f"]
        edges = sp["edges"]
        col_z = z_imp[:, j]
        is_miss = np.isnan(X_sub_nan[:, j]) if j < 20 else np.zeros(n_rows, dtype=bool)
        for b in range(len(edges) - 1):
            denom = max(1e-5, edges[b + 1] - edges[b])
            v = np.clip((col_z - edges[b]) / denom, 0.0, 1.0)
            if np.any(is_miss):
                v[is_miss] = 0.0
            ple_cols.append(v[:, None].astype(np.float32))
    return np.hstack([slog, miss20] + ple_cols)

ple_specs = fit_ple_specs(X_tr_nan, n_bins=6)
X_tr_ple = encode_ple_matrix(X_tr_nan, ple_specs)
X_va_ple = encode_ple_matrix(X_va_nan, ple_specs)
X_te_ple = encode_ple_matrix(X_te_nan, ple_specs)
D_IN = int(X_tr_ple.shape[1])
print(f"PLE Encoded Input Dim: D={D_IN} (79 signed-log1p + 20 miss masks + {D_IN - 99} quantile bins across {len(ple_specs)} vars)")

# 4. PyTorch Neural Architectures: SwiGLUNet (Comparison) & 5-Fold PLE-ResNet (Primary)
class SwiGLUNet(nn.Module):
    def __init__(self, d_in, hid=64, drop=0.25):
        super().__init__()
        self.in_proj = nn.Linear(d_in, hid * 2)
        self.ln1 = nn.LayerNorm(hid, eps=1e-5)
        self.blk = nn.Linear(hid, hid * 2)
        self.ln2 = nn.LayerNorm(hid, eps=1e-5)
        self.head = nn.Linear(hid, 1)
        self.skip = nn.Linear(d_in, 1)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        a, b = self.in_proj(x).chunk(2, dim=-1)
        h = self.ln1(F.silu(a) * b)
        c, d = self.blk(self.drop(h)).chunk(2, dim=-1)
        h = h + self.ln2(F.silu(c) * d)
        return (self.head(h) + 0.5 * self.skip(x)).squeeze(-1)

class PLEResNet(nn.Module):
    def __init__(self, d_in, hid=64, drop=0.22):
        super().__init__()
        self.stem_fc = nn.Linear(d_in, hid)
        self.stem_ln = nn.LayerNorm(hid, eps=1e-5)
        self.b1_fc1 = nn.Linear(hid, hid)
        self.b1_ln = nn.LayerNorm(hid, eps=1e-5)
        self.b1_fc2 = nn.Linear(hid, hid)
        self.b2_fc1 = nn.Linear(hid, hid)
        self.b2_ln = nn.LayerNorm(hid, eps=1e-5)
        self.b2_fc2 = nn.Linear(hid, hid)
        self.head = nn.Linear(hid, 1)
        self.skip = nn.Linear(d_in, 1)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        h = F.silu(self.stem_ln(self.stem_fc(x)))
        h = h + self.b1_fc2(self.drop(F.silu(self.b1_ln(self.b1_fc1(h)))))
        h = h + self.b2_fc2(self.drop(F.silu(self.b2_ln(self.b2_fc1(h)))))
        return (self.head(h) + 0.5 * self.skip(x)).squeeze(-1)

def predict_torch_logits(model, X_np, batch_size=4096):
    model.eval()
    outs = []
    with torch.no_grad():
        for i in range(0, len(X_np), batch_size):
            xb = torch.from_numpy(X_np[i : i + batch_size]).to(DEVICE)
            outs.append(model(xb).detach().cpu().numpy())
    return np.concatenate(outs)

def train_neural_candidate(model_cls, X_ft, y_ft, X_fv, y_fv, seed=42, epochs=18, lr=2.5e-3, wd=2.5e-4, batch_size=2048):
    torch.manual_seed(seed)
    model = model_cls(X_ft.shape[1], hid=64).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-4)

    n_pos = max(1, int(np.sum(y_ft == 1)))
    n_neg = max(1, int(np.sum(y_ft == 0)))
    pos_w = torch.tensor([min(12.0, math.sqrt(n_neg / n_pos))], dtype=torch.float32, device=DEVICE)

    Xt_t = torch.from_numpy(X_ft).to(DEVICE)
    yt_t = torch.from_numpy(y_ft.astype(np.float32)).to(DEVICE)

    best_auc, best_state = -1.0, None
    n_rows = len(X_ft)
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n_rows, device=DEVICE)
        for s_i in range(0, n_rows, batch_size):
            idx = perm[s_i : s_i + batch_size]
            xb, yb = Xt_t[idx], yt_t[idx]
            logits = model(xb)
            bce = F.binary_cross_entropy_with_logits(logits, yb, pos_weight=pos_w)
            pos_l = logits[yb > 0.5]
            neg_l = logits[yb <= 0.5][:512]
            if len(pos_l) > 0 and len(neg_l) > 0:
                auc_loss = F.softplus(-(pos_l.unsqueeze(1) - neg_l.unsqueeze(0))).mean()
                loss = 0.55 * bce + 0.45 * auc_loss
            else:
                loss = bce
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step()
        sched.step()
        v_logits = predict_torch_logits(model, X_fv)
        v_auc = float(roc_auc_score(y_fv, _sig(v_logits)))
        if v_auc > best_auc:
            best_auc = v_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    model.to(DEVICE)
    return model, best_auc

# Train SwiGLUNet comparison (on Train -> Val early stop)
t_nn0 = time.time()
m_swiglu, swiglu_val_auc = train_neural_candidate(SwiGLUNet, X_tr_ple, y_tr, X_va_ple, y_va, seed=42, epochs=16)
z_swi_va = predict_torch_logits(m_swiglu, X_va_ple)
z_swi_te = predict_torch_logits(m_swiglu, X_te_ple)
print(f"SwiGLUNet (Neural Comparison): Val AUROC={roc_auc_score(y_va, _sig(z_swi_va)):.4f}")

# Train 5-Fold StratifiedGroupKFold PLE-ResNet (Primary Model) strictly inside Train (zero Val/Test leakage)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
nn_models, fold_val_aucs = [], []
oof_tr_logits = np.zeros(len(y_tr), dtype=np.float64)
val_fold_logits, test_fold_logits = [], []

for f_i, (ti, vi) in enumerate(sgkf.split(X_tr_ple, y_tr, groups=g_tr), 1):
    m_fold, f_auc = train_neural_candidate(
        PLEResNet, X_tr_ple[ti], y_tr[ti], X_tr_ple[vi], y_tr[vi], seed=100 + f_i * 17, epochs=18
    )
    nn_models.append(m_fold)
    fold_val_aucs.append(round(f_auc, 4))
    oof_tr_logits[vi] = predict_torch_logits(m_fold, X_tr_ple[vi])
    val_fold_logits.append(predict_torch_logits(m_fold, X_va_ple))
    test_fold_logits.append(predict_torch_logits(m_fold, X_te_ple))
    print(f"  [PLE-ResNet Fold {f_i}/5] fold_val AUROC={f_auc:.4f}")

z_nn_va = np.mean(val_fold_logits, axis=0)
z_nn_te = np.mean(test_fold_logits, axis=0)
nn_train_sec = time.time() - t_nn0

# Fit 1D Platt Sigmoid Calibrator strictly on Validation logits
platt_lr = LogisticRegression(C=10.0, max_iter=500).fit(z_nn_va.reshape(-1, 1), y_va)
platt_a = float(platt_lr.coef_[0][0])
platt_b = float(platt_lr.intercept_[0])
print(f"5-Fold PLE-ResNet trained in {nn_train_sec:.1f}s | Train OOF AUROC={roc_auc_score(y_tr, _sig(oof_tr_logits)):.4f} | Mean Fold={np.mean(fold_val_aucs):.4f} +/- {np.std(fold_val_aucs):.4f}")
print(f"5-Fold PLE-ResNet Val AUROC={roc_auc_score(y_va, _sig(z_nn_va)):.4f} | Platt: a={platt_a:.4f}, b={platt_b:.4f}")


### Validation & 5-Fold Cross-Validation Analysis
Compare single-model baselines against the **5-Fold `StratifiedGroupKFold PLE-ResNet`** above:
- Notice why `PLE + Missingness Masks + Signed-Log1p` unlocks strong neural generalization: unmeasured labs remain cleanly isolated (`mask=1`, `PLE=0`) instead of injecting artificial cohort means (`Lactate=2.47`) into linear layers.
- Pairwise AUROC ranking loss optimizes hourly risk ordering directly, 5-fold patient-grouped bagging (`mean +/- SD` above) stabilizes variance across unseen cohorts, and 1D Platt Sigmoid Calibration on untouched `Val` transforms balanced logits into honest 6-hour onset probabilities.


## 7. Held-Out Test Monitoring Dashboard: Discrimination, Calibration, Utility & Alarm Burden
Evaluates all five models on the completely untouched **Held-Out Test Patients** (`y_te`) using operating thresholds locked on Validation, plus **400-resample patient-bootstrap 95% confidence intervals**.


In [ ]:
def fit_platt_prob(z_v, y_va_in, z_t):
    cal = LogisticRegression(C=10.0, max_iter=400).fit(z_v.reshape(-1, 1), y_va_in)
    return _sig(cal.coef_[0][0] * z_v + cal.intercept_[0]), _sig(cal.coef_[0][0] * z_t + cal.intercept_[0])

p_clin_va, p_clin_te = fit_platt_prob(_logit(m_clin.predict_proba(X_va_imp[:, clin_idx])[:, 1]), y_va, _logit(m_clin.predict_proba(X_te_imp[:, clin_idx])[:, 1]))
p_lr_va, p_lr_te     = fit_platt_prob(_logit(m_lr.predict_proba(X_va_imp)[:, 1]), y_va, _logit(m_lr.predict_proba(X_te_imp)[:, 1]))
p_hg_va, p_hg_te     = fit_platt_prob(_logit(m_hgb.predict_proba(X_va_nan)[:, 1]), y_va, _logit(m_hgb.predict_proba(X_te_nan)[:, 1]))
p_swi_va, p_swi_te   = fit_platt_prob(z_swi_va, y_va, z_swi_te)

p_nn_va = _sig(platt_a * z_nn_va + platt_b)
p_nn_te = _sig(platt_a * z_nn_te + platt_b)

eval_pairs = [
    ("NEWS2/qSOFA Baseline", p_clin_va, p_clin_te),
    (f"L2 LogReg (C={best_c})", p_lr_va, p_lr_te),
    ("HistGB (Reference)", p_hg_va, p_hg_te),
    ("SwiGLUNet (Neural)", p_swi_va, p_swi_te),
    ("5-Fold PLE-ResNet (Calibrated NN)", p_nn_va, p_nn_te),
]

rows_tbl = []
test_results = {}
val_results = {}
skip_keys = {"probs", "pt_y", "pt_p", "pt_2of3_preds", "threshold_raw", "watchlist_th_raw"}
for label, p_v, p_t in eval_pairs:
    val_res = evaluate_monitoring_suite(y_va, p_v, g_va, threshold=None, watchlist_threshold=None, max_fp_per_100h=5.0)
    te_res = evaluate_monitoring_suite(
        y_te, p_t, g_te,
        threshold=val_res["threshold_raw"],
        watchlist_threshold=val_res["watchlist_th_raw"],
    )
    val_results[label] = val_res
    test_results[label] = te_res
    row = {"Model": label}
    for k, v in te_res.items():
        if k not in skip_keys:
            row[k] = v
    rows_tbl.append(row)

monitor_df = pd.DataFrame(rows_tbl)
display(monitor_df)

# Print Two-Tier Bedside Deployment Summary for 5-Fold PLE-ResNet
nn_key = "5-Fold PLE-ResNet (Calibrated NN)"
nn_te = test_results[nn_key]
nn_th = nn_te["threshold_raw"]
nn_watch_th = nn_te["watchlist_th_raw"]
w_tn, w_fp, w_fn, w_tp = confusion_matrix(y_te, (p_nn_te >= nn_watch_th).astype(int), labels=[0, 1]).ravel()
c_tn, c_fp, c_fn, c_tp = confusion_matrix(y_te, (p_nn_te >= nn_th).astype(int), labels=[0, 1]).ravel()
p_tn, p_fp, p_fn, p_tp = confusion_matrix(nn_te["pt_y"], nn_te["pt_2of3_preds"], labels=[0, 1]).ravel()
policy_df = pd.DataFrame([
    {"Deployment Tier": f"1. Passive Watchlist (>= {nn_watch_th*100:.2f}%)", "Granularity": "Hourly Stream", "TP": int(w_tp), "FP": int(w_fp), "FP_per_100h": round(100.0 * w_fp / len(y_te), 2), "Specificity": round(w_tn / max(1, w_tn + w_fp), 4), "PPV": round(w_tp / max(1, w_tp + w_fp), 4), "Recall/Catch": nn_te["Watch_24h"]},
    {"Deployment Tier": f"2. Critical Alert Budget (>= {nn_th*100:.2f}%, Val<=5/100h)", "Granularity": "Hourly Stream", "TP": int(c_tp), "FP": int(c_fp), "FP_per_100h": round(100.0 * c_fp / len(y_te), 2), "Specificity": round(c_tn / max(1, c_tn + c_fp), 4), "PPV": round(c_tp / max(1, c_tp + c_fp), 4), "Recall/Catch": nn_te["Caught_24h"]},
    {"Deployment Tier": f"3. Sustained 2-of-3h Critical (>= {nn_th*100:.2f}%)", "Granularity": "Patient Stay", "TP": int(p_tp), "FP": int(p_fp), "FP_per_100h": np.nan, "Specificity": round(p_tn / max(1, p_tn + p_fp), 4), "PPV": round(p_tp / max(1, p_tp + p_fp), 4), "Recall/Catch": f"{int(p_tp)}/{int(p_tp + p_fn)}"},
])
print("Two-Tier Bedside Alert Policy Comparison (Held-Out Test):")
display(policy_df)

# Bootstrap 95% CI (Patient Cluster Bootstrap on Held-Out Test)
rng_bs = np.random.default_rng(42)
unique_te_pts = np.unique(g_te)
pt_rows_idx = {pid: np.flatnonzero(g_te == pid) for pid in unique_te_pts}
bs_aucs, bs_prs = [], []
for _ in range(400):
    samp_pts = rng_bs.choice(unique_te_pts, size=len(unique_te_pts), replace=True)
    idx_s = np.concatenate([pt_rows_idx[pid] for pid in samp_pts])
    if len(np.unique(y_te[idx_s])) == 2:
        bs_aucs.append(roc_auc_score(y_te[idx_s], p_nn_te[idx_s]))
        bs_prs.append(average_precision_score(y_te[idx_s], p_nn_te[idx_s]))
print(f"5-Fold PLE-ResNet Test 95% Bootstrap CI -> AUROC: [{np.percentile(bs_aucs, 2.5):.4f}, {np.percentile(bs_aucs, 97.5):.4f}] | AUPRC: [{np.percentile(bs_prs, 2.5):.4f}, {np.percentile(bs_prs, 97.5):.4f}]")

# Visual Monitoring Panel (2x3 Deployment Dashboard)
fig, axes = plt.subplots(2, 3, figsize=(18.5, 10.5))
hr_prev = float(np.mean(y_te))
pt_prev = float(np.mean(nn_te["pt_y"]))
pr_y_max = 0.0

for label, _, _ in eval_pairs:
    is_primary = (label == nn_key)
    lw = 2.6 if is_primary else 1.5
    zo = 5 if is_primary else 2
    alpha_c = 1.0 if is_primary else 0.78

    # Hourly ROC & PR
    pr = test_results[label]["probs"]
    fpr, tpr, _ = roc_curve(y_te, pr)
    prec, rec, _ = precision_recall_curve(y_te, pr)
    axes[0, 0].plot(fpr, tpr, lw=lw, alpha=alpha_c, zorder=zo, label=f"{label} ({test_results[label]['Hourly AUROC']:.3f})")
    axes[0, 1].plot(rec, prec, lw=lw, alpha=alpha_c, zorder=zo, label=f"{label} ({test_results[label]['AUPRC']:.3f})")
    valid_pr = prec[rec >= 0.02]
    if len(valid_pr) > 0:
        pr_y_max = max(pr_y_max, float(np.max(valid_pr)))

    # Patient-Episode PR (Max Stay Risk)
    prec_p, rec_p, _ = precision_recall_curve(test_results[label]["pt_y"], test_results[label]["pt_p"])
    axes[0, 2].plot(rec_p, prec_p, lw=lw, alpha=alpha_c, zorder=zo, label=f"{label} ({test_results[label]['Pt_AUPRC']:.3f})")

axes[0, 0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0, 0].set_title(f"Held-Out Test ROC Curve (Pt AUROC={nn_te['Patient AUROC']:.3f})")
axes[0, 0].set_xlabel("False Positive Rate (1 - Specificity)"); axes[0, 0].set_ylabel("True Positive Rate (Sensitivity)")
axes[0, 0].legend(fontsize=8, loc="lower right")

axes[0, 1].axhline(hr_prev, color="#475569", linestyle="--", lw=1.2, label=f"Random Baseline ({hr_prev*100:.2f}%)")
axes[0, 1].set_ylim(0.0, min(0.45, max(0.28, pr_y_max * 1.18)))
axes[0, 1].set_title(f"Hourly PR Curve (NN Lift = {nn_te['AUPRC']/max(1e-6, hr_prev):.1f}x Baseline)")
axes[0, 1].set_xlabel("Recall (Sensitivity)"); axes[0, 1].set_ylabel("Hourly Precision (PPV)")
axes[0, 1].legend(fontsize=8, loc="upper right")

axes[0, 2].axhline(pt_prev, color="#475569", linestyle="--", lw=1.2, label=f"Patient Incidence ({pt_prev*100:.1f}%)")
axes[0, 2].set_ylim(0.0, 1.02)
axes[0, 2].set_title(f"Patient-Episode PR Curve (NN Pt Lift = {nn_te['Pt_AUPRC']/max(1e-6, pt_prev):.1f}x)")
axes[0, 2].set_xlabel("Patient Recall (Sepsis+ Stays)"); axes[0, 2].set_ylabel("Patient Precision (PPV)")
axes[0, 2].legend(fontsize=8, loc="upper right")

# Calibration Reliability Diagram: 10 Equal-Count Quantile Deciles + Full-Range Uncalibrated Inset
uncal_nn_te = _sig(z_nn_te)
order_cal = np.argsort(p_nn_te)
dec_chunks = np.array_split(order_cal, 10)
dec_x = [float(np.mean(p_nn_te[ch])) for ch in dec_chunks]
dec_y = [float(np.mean(y_te[ch])) for ch in dec_chunks]
q_lim = min(1.0, max(0.14, max(max(dec_x), max(dec_y)) * 1.25))

axes[1, 0].plot(dec_x, dec_y, "o-", lw=2.3, ms=6, color="#0284c7", label="Platt Calibrated NN (10 Quantile Deciles)")
axes[1, 0].plot([0, q_lim], [0, q_lim], "k--", alpha=0.55, label="Perfect Calibration (y = x)")
axes[1, 0].set_xlim(0.0, q_lim); axes[1, 0].set_ylim(0.0, q_lim)
axes[1, 0].set_title(f"Quantile Calibration Diagram (Brier={nn_te['Brier']:.4f}, ECE={nn_te['ECE']:.4f})")
axes[1, 0].set_xlabel("Mean Predicted 6h Probability"); axes[1, 0].set_ylabel("Observed Sepsis Rate")
axes[1, 0].legend(fontsize=8, loc="upper left")

ax_ins = axes[1, 0].inset_axes([0.55, 0.12, 0.41, 0.37])
bins_u = np.linspace(0, 1, 11)
ux, uy = [], []
for bi in range(10):
    mb = (uncal_nn_te >= bins_u[bi]) & (uncal_nn_te <= bins_u[bi + 1])
    if np.sum(mb) >= 10:
        ux.append(float(np.mean(uncal_nn_te[mb]))); uy.append(float(np.mean(y_te[mb])))
ax_ins.plot(ux, uy, "o-", lw=1.4, ms=3.5, color="#94a3b8", label="Uncalibrated NN")
ax_ins.plot(dec_x, dec_y, "-", lw=2.0, color="#0284c7", label="Platt Calibrated")
ax_ins.plot([0, 1], [0, 1], "k--", lw=0.9, alpha=0.5)
ax_ins.set_xlim(0, 1); ax_ins.set_ylim(0, 1)
ax_ins.set_title("Full [0, 1] Range", fontsize=7.5)
ax_ins.tick_params(labelsize=7)
ax_ins.legend(fontsize=6.5, loc="upper left")

def _annot_cm(cm_arr):
    labels = np.empty_like(cm_arr, dtype=object)
    for r in range(2):
        row_s = max(1, int(np.sum(cm_arr[r])))
        for c_i in range(2):
            labels[r, c_i] = f"{cm_arr[r, c_i]:,}\n({100.0 * cm_arr[r, c_i] / row_s:.1f}%)"
    return labels

# Hourly Confusion Matrix at Alarm-Budgeted Deployment Threshold (Val <= 5.0 FP/100h)
cm_hr = confusion_matrix(y_te, (p_nn_te >= nn_th).astype(int), labels=[0, 1])
sns.heatmap(cm_hr, annot=_annot_cm(cm_hr), fmt="", cmap="Blues", ax=axes[1, 1], xticklabels=["Pred 0", "Pred Alert"], yticklabels=["True 0", "True 1"], cbar=False)
axes[1, 1].set_title(f"Hourly Alert Matrix (Crit >= {nn_th*100:.2f}%, {nn_te['FP_per_100h']} FP/100h)")

# Patient-Episode Confusion Matrix with 2-of-3h Sustained Persistence
cm_pt = confusion_matrix(nn_te["pt_y"], nn_te["pt_2of3_preds"], labels=[0, 1])
sns.heatmap(cm_pt, annot=_annot_cm(cm_pt), fmt="", cmap="BuGn", ax=axes[1, 2], xticklabels=["No Stay Alert", "2-of-3h Alert"], yticklabels=["Control Stay", "Sepsis+ Stay"], cbar=False)
axes[1, 2].set_title(f"Patient 2-of-3h Sustained Matrix (Spec={nn_te['Pt_2of3_Spec']*100:.1f}%, PPV={nn_te['Pt_2of3_PPV']*100:.1f}%)")
plt.tight_layout()
plt.show()


### Monitoring Dashboard Analysis
Inspect the six deployment dimensions above:
- **Discrimination (`ROC`, `Hourly PR`, `Patient-Episode PR`):** `5-Fold PLE-ResNet` leads across both hourly predictions and full ICU stays, delivering strong lift above the natural `~2.2%` hourly and `~9%` patient sepsis baselines.
- **Quantile Reliability (`Brier`, `ECE`):** Plotting **10 equal-count quantile deciles** verifies that calibrated 6-hour risk scores (`0.3%` up to `8%+`) closely track actual observed sepsis rates along `y = x`, while the `[0, 1]` inset explains why raw class-weighted neural logits needed Platt scaling.
- **Bedside Alarm Control (`Hourly Matrix` & `2-of-3h Patient Matrix`):** Locking the **Critical Alert** on `Validation` to `<= 5.0 FP/100h` (`>= 95%` hourly specificity) cuts false hourly alarms by ~75% vs unconstrained utility thresholding, and requiring **2-of-3 consecutive elevated hours** raises patient-level specificity (`~90-96%`) and actionable PPV (`~30-45%`) for real ward deployment.


## 8. Lead-Time Horizon, Alarm Trade-Off Curve, Subgroup Slices & Neural Saliency / Permutation Importance
Audits **how early** alerts fire prior to `t_sepsis`, verifies performance across **Age & Early-vs-Late Sepsis subgroups**, and ranks the top predictive signals directly from the **5-Fold `PLE-ResNet` Neural Network** via Input-Gradient Saliency and Held-Out Test Permutation Importance.


In [ ]:
# 1. Early Lead-Time Distribution & Utility/FP Trade-Off Curve
df_te_eval = pd.DataFrame({"patient_id": g_te, "ICULOS": df_raw.loc[te_mask, "ICULOS"].to_numpy(), "y": y_te, "p": p_nn_te})
lead_list = []
for pid, sub in df_te_eval.groupby("patient_id", sort=False):
    ys = sub["y"].to_numpy(); ps = sub["p"].to_numpy()
    if np.max(ys) == 1:
        t_on = int(np.argmax(ys == 1)) + 6
        idx24 = np.flatnonzero(ps[max(0, t_on - 24) : min(len(ps), t_on + 4)] >= nn_th)
        if len(idx24) > 0:
            lead_list.append(t_on - (max(0, t_on - 24) + int(idx24[0])))

th_grid = np.quantile(p_nn_te, np.linspace(0.60, 0.99, 25))
curve_pts = []
te_groups = [(pid, sub["y"].to_numpy(), sub["p"].to_numpy()) for pid, sub in df_te_eval.groupby("patient_id", sort=False)]
for th_c in np.unique(np.round(th_grid, 4)):
    us = uo_s = un_s = 0.0
    for _, ys, ps in te_groups:
        u, uo, un = physionet_utility_single(ys, (ps >= th_c).astype(int))
        us += u; uo_s += uo; un_s += un
    pr_b = (p_nn_te >= th_c).astype(int)
    sens_c = float(np.sum((y_te == 1) & (pr_b == 1)) / max(1, np.sum(y_te == 1)))
    fp100_c = float(100.0 * np.sum((y_te == 0) & (pr_b == 1)) / len(y_te))
    curve_pts.append({"th": th_c, "util": (us - un_s) / max(1e-9, uo_s - un_s), "sens": sens_c, "fp100": fp100_c})
c_df = pd.DataFrame(curve_pts)

# 2. Subgroup Slice Audit (Age >= 65 vs < 65, ICU stay <= 20h vs > 20h)
age_te = df_raw.loc[te_mask, "Age"].to_numpy()
ic_te = df_raw.loc[te_mask, "ICULOS"].to_numpy()
slices = [
    ("Age < 65 yrs", age_te < 65),
    ("Age >= 65 yrs", age_te >= 65),
    ("Early ICU (<= 20h)", ic_te <= 20),
    ("Late ICU (> 20h)", ic_te > 20),
]
slice_rows = []
for sname, smask in slices:
    if len(np.unique(y_te[smask])) == 2:
        slice_rows.append({
            "Subgroup": sname,
            "Rows": int(np.sum(smask)),
            "PosRows": int(np.sum(y_te[smask])),
            "AUROC": round(float(roc_auc_score(y_te[smask], p_nn_te[smask])), 4),
            "AUPRC": round(float(average_precision_score(y_te[smask], p_nn_te[smask])), 4),
        })
display(pd.DataFrame(slice_rows))

# 3. Neural Input-Gradient Saliency & Test Permutation Importance (directly on 5-Fold PLE-ResNet)
x_sal = torch.from_numpy(X_te_ple[:4096].copy()).to(DEVICE).requires_grad_(True)
sal_logits = sum(m(x_sal) for m in nn_models) / len(nn_models)
sal_logits.sum().backward()
raw_grad = x_sal.grad.detach().abs().mean(dim=0).cpu().numpy()
sal_by_feat = raw_grad[:len(FEATURES)].copy()
for k_i in range(20):
    sal_by_feat[k_i] += raw_grad[len(FEATURES) + k_i]
offset_b = len(FEATURES) + 20
for sp in ple_specs:
    n_b = len(sp["edges"]) - 1
    sal_by_feat[sp["f"]] += float(np.sum(raw_grad[offset_b : offset_b + n_b]))
    offset_b += n_b
saliency_top15 = [
    {"feature": FEATURES[i], "mean_abs_grad": round(float(sal_by_feat[i]), 5)}
    for i in np.argsort(-sal_by_feat)[:15]
]

base_auc = float(roc_auc_score(y_te, _sig(z_nn_te)))
cand_feats = sorted(set(np.argsort(-sal_by_feat)[:25].tolist() + [0, 1, 2, 3, 4, 10, 11, 12, 22, 40, 77, 78]))
rng_p = np.random.default_rng(42)
perm_rows = []
for f_idx in cand_feats:
    drops = []
    for _ in range(3):
        X_perm = X_te_nan.copy()
        X_perm[:, f_idx] = rng_p.permutation(X_perm[:, f_idx])
        Xp_ple = encode_ple_matrix(X_perm, ple_specs)
        zp = np.mean([predict_torch_logits(m, Xp_ple, batch_size=8192) for m in nn_models], axis=0)
        drops.append(base_auc - float(roc_auc_score(y_te, _sig(zp))))
    perm_rows.append({"Feature": FEATURES[f_idx], "AUROC_Drop": float(np.mean(drops))})
imp_df = pd.DataFrame(perm_rows).sort_values("AUROC_Drop", ascending=False).head(15)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
axes[0].hist(lead_list, bins=range(-3, 26), color="#0284c7", edgecolor="#0f172a", alpha=0.85)
axes[0].axvline(6, color="#b91c1c", linestyle="--", label="Target 6h Forecast Horizon")
axes[0].set_title(f"Critical Lead Time (>= {nn_th*100:.2f}%)")
axes[0].set_xlabel("Hours Before Clinical Sepsis-3 Onset"); axes[0].set_ylabel("Detected Sepsis+ Patients")
axes[0].legend()

axes[1].plot(c_df["fp100"], c_df["util"], "o-", color="#008775", lw=2, label="Normalized PhysioNet Utility")
axes[1].axvline(5.0, color="#b91c1c", linestyle="--", lw=1.4, label="Critical Alarm Budget (5 FP/100h)")
axes[1].set_title("Clinical Utility vs Alarm Fatigue")
axes[1].set_xlabel("False Alarms per 100 Patient-Hours"); axes[1].set_ylabel("Normalized Utility Score")
axes[1].legend(fontsize=8)

imp_p = imp_df.sort_values("AUROC_Drop", ascending=True)
axes[2].barh(imp_p["Feature"], imp_p["AUROC_Drop"], color="#0284c7")
axes[2].set_title("Top 15 Neural Predictors (Test Permutation AUROC Drop)")
axes[2].set_xlabel("Mean AUROC Drop when Permuted")
plt.tight_layout()
plt.show()


### Operational & Clinical Explainability Summary
- **Lead Time & Alarm Budget:** Check where the green **Normalized PhysioNet Utility** peaks relative to `False Alarms per 100 Patient-Hours`, and verify via the subgroup table that discrimination stays stable across elderly (`>= 65 yrs`) and late-onset ICU patients (`> 20h`).
- **Top Neural Predictors:** Both Input-Gradient Saliency and Held-Out Test Permutation Importance confirm that the 5-Fold `PLE-ResNet` relies on clinically coherent signals (`ICULOS`, `Lactate`, `Lab_update_events`, `O2Sat_min_6h`, `HospAdmTime`, `WBC_high_hinge`, `Platelets_delta`, `Resp_diff_base`).


## 9. Export Web-App Compatible `scaler.json` and `predictive_model.json`
Serializes the fitted `StandardScaler`, 28 `ple_specs`, all **5 PyTorch `PLE-ResNet` folds** (`stem_*`, `b1_*`, `b2_*`, `head_*`, `skip_*`), Platt calibration `(a, b)`, L2 LogReg comparison reference, and full monitoring telemetry - verifying exact PyTorch-to-JSON forward numerical parity (`< 1e-4`) before saving.


In [ ]:
def _r1d(t):
    return [round(float(x), 5) for x in t.detach().cpu().numpy().reshape(-1)]

def _r2d(t):
    return [[round(float(x), 5) for x in row] for row in t.detach().cpu().numpy()]

exported_folds = []
for f_idx, m_fold in enumerate(nn_models):
    sd_f = m_fold.state_dict()
    exported_folds.append({
        "fold_val_auroc": fold_val_aucs[f_idx],
        "stem_w": _r2d(sd_f["stem_fc.weight"]),
        "stem_b": _r1d(sd_f["stem_fc.bias"]),
        "stem_ln_g": _r1d(sd_f["stem_ln.weight"]),
        "stem_ln_b": _r1d(sd_f["stem_ln.bias"]),
        "b1_fc1_w": _r2d(sd_f["b1_fc1.weight"]),
        "b1_fc1_b": _r1d(sd_f["b1_fc1.bias"]),
        "b1_ln_g": _r1d(sd_f["b1_ln.weight"]),
        "b1_ln_b": _r1d(sd_f["b1_ln.bias"]),
        "b1_fc2_w": _r2d(sd_f["b1_fc2.weight"]),
        "b1_fc2_b": _r1d(sd_f["b1_fc2.bias"]),
        "b2_fc1_w": _r2d(sd_f["b2_fc1.weight"]),
        "b2_fc1_b": _r1d(sd_f["b2_fc1.bias"]),
        "b2_ln_g": _r1d(sd_f["b2_ln.weight"]),
        "b2_ln_b": _r1d(sd_f["b2_ln.bias"]),
        "b2_fc2_w": _r2d(sd_f["b2_fc2.weight"]),
        "b2_fc2_b": _r1d(sd_f["b2_fc2.bias"]),
        "head_w": _r1d(sd_f["head.weight"]),
        "head_b": round(float(sd_f["head.bias"].item()), 5),
        "skip_w": _r1d(sd_f["skip.weight"]),
        "skip_b": round(float(sd_f["skip.bias"].item()), 5),
    })

def _ln_np(x, g, b, eps=1e-5):
    mu = np.mean(x, axis=-1, keepdims=True)
    var = np.mean((x - mu) ** 2, axis=-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps) * g + b

def _silu_np(x):
    return x / (1.0 + np.exp(-np.clip(x, -20.0, 20.0)))

def eval_exported_nn_numpy(X_ple_sub, nn_cfg):
    eps = nn_cfg["layernorm_eps"]
    skip_scale = nn_cfg["skip_scale"]
    fold_sum = np.zeros(len(X_ple_sub), dtype=np.float64)
    for fw in nn_cfg["folds"]:
        stem = X_ple_sub @ np.asarray(fw["stem_w"], dtype=np.float64).T + np.asarray(fw["stem_b"])
        h = _silu_np(_ln_np(stem, np.asarray(fw["stem_ln_g"]), np.asarray(fw["stem_ln_b"]), eps))
        r1p = h @ np.asarray(fw["b1_fc1_w"]).T + np.asarray(fw["b1_fc1_b"])
        r1a = _silu_np(_ln_np(r1p, np.asarray(fw["b1_ln_g"]), np.asarray(fw["b1_ln_b"]), eps))
        h = h + r1a @ np.asarray(fw["b1_fc2_w"]).T + np.asarray(fw["b1_fc2_b"])
        r2p = h @ np.asarray(fw["b2_fc1_w"]).T + np.asarray(fw["b2_fc1_b"])
        r2a = _silu_np(_ln_np(r2p, np.asarray(fw["b2_ln_g"]), np.asarray(fw["b2_ln_b"]), eps))
        h = h + r2a @ np.asarray(fw["b2_fc2_w"]).T + np.asarray(fw["b2_fc2_b"])
        head = h @ np.asarray(fw["head_w"]) + fw["head_b"]
        skip = X_ple_sub @ np.asarray(fw["skip_w"]) + fw["skip_b"]
        fold_sum += head + skip_scale * skip
    return fold_sum / len(nn_cfg["folds"])

n_params_per_fold = sum(p.numel() for p in nn_models[0].parameters())
nn_block = {
    "architecture": "5-Fold PLE-ResNet (LayerNorm + SiLU + Residual Skip, hid=64)",
    "input_dim": D_IN,
    "hidden_dim": 64,
    "n_folds": len(exported_folds),
    "n_params_per_fold": int(n_params_per_fold),
    "layernorm_eps": 1e-5,
    "skip_scale": 0.5,
    "ple_specs": ple_specs,
    "folds": exported_folds,
    "top_saliency_features": saliency_top15,
}

# Verify exact parity between PyTorch 5-Fold ensemble and exported JSON rounded weights
json_logits_chk = eval_exported_nn_numpy(X_te_ple[:80], nn_block)
max_diff = float(np.max(np.abs(z_nn_te[:80] - json_logits_chk)))
assert max_diff < 1e-3, f"Neural JSON export mismatch: {max_diff}"
print(f"Verified PyTorch-to-JSON 5-Fold Neural Network parity on 80 test rows (max logit diff={max_diff:.6f})")

# Export scaler.json in physical bedside units over core raw columns
scaler_out = {}
for col in [c for c in CORE_TELEMETRY if c in df_raw.columns]:
    s = df_raw[col].dropna()
    if len(s) > 1:
        scaler_out[col] = {"mean": round(float(s.mean()), 4), "sd": round(float(s.std()), 4), "n_obs": int(len(s))}

with open(os.path.join(OUTPUT_DIR, "scaler.json"), "w") as f:
    json.dump(scaler_out, f, indent=2)

def _clean_m(d):
    skip = {"probs", "pt_y", "pt_p", "pt_2of3_preds", "threshold_raw", "watchlist_th_raw"}
    return {
        k.lower().replace(" ", "_"): v
        for k, v in d.items()
        if k not in skip
    }

val_m = val_results[nn_key]
te_m = test_results[nn_key]
lr_m = test_results[f"L2 LogReg (C={best_c})"]

model_artifact = {
    "model_name": f"Kaggle 5-Fold PLE-ResNet Neural Network ({len(FEATURES)} causal -> {D_IN}D PLE)",
    "input_scale": "zscore" if IS_ZSCORED else "raw_clinical_units",
    "cohort_summary": {
        "total_patients": int(n_patients),
        "train_patients": int(len(train_pts)),
        "val_patients": int(len(val_pts)),
        "test_patients": int(len(test_pts)),
        "total_hours": int(len(df_raw)),
        "sepsis_positive_patients": int(pt_stats["has_sepsis"].sum()),
    },
    "features": FEATURES,
    "scaler_means": {f: round(float(m_v), 5) for f, m_v in zip(FEATURES, train_means)},
    "scaler_sds": {f: round(float(s_v), 5) for f, s_v in zip(FEATURES, train_sds)},
    "neural_network": nn_block,
    "logistic_regression": {
        "l2_reg": best_c,
        "bias": round(float(m_lr.intercept_[0]), 6),
        "weights": [round(float(w), 6) for w in m_lr.coef_[0]],
    },
    "calibration": {"method": "platt_sigmoid_val", "use_logit_blend": False, "a": round(platt_a, 6), "b": round(platt_b, 6)},
    "ensemble": {
        "neural_weight": 1.0,
        "logreg_weight": 0.0,
        "watchlist_threshold": round(float(val_m["watchlist_th_raw"]), 4),
        "decision_threshold": round(float(val_m["threshold_raw"]), 4),
        "val_metrics": {
            "auroc": val_m["Hourly AUROC"],
            "patient_auroc": val_m["Patient AUROC"],
            "auprc": val_m["AUPRC"],
            "pt_auprc": val_m["Pt_AUPRC"],
            "physionet_utility": val_m["Utility Score"],
            "brier_score": val_m["Brier"],
            "ece": val_m["ECE"],
            "sensitivity": val_m["Sensitivity"],
            "specificity": val_m["Specificity"],
        },
        "test_metrics": {
            "auroc": te_m["Hourly AUROC"],
            "patient_auroc": te_m["Patient AUROC"],
            "auprc": te_m["AUPRC"],
            "pt_auprc": te_m["Pt_AUPRC"],
            "physionet_utility": te_m["Utility Score"],
            "brier_score": te_m["Brier"],
            "ece": te_m["ECE"],
            "sensitivity": te_m["Sensitivity"],
            "specificity": te_m["Specificity"],
            "fp_per_100h": te_m["FP_per_100h"],
            "pt_2of3_spec": te_m["Pt_2of3_Spec"],
            "pt_2of3_ppv": te_m["Pt_2of3_PPV"],
        },
    },
    "monitoring_metrics": {
        "train_oof_auroc": round(float(roc_auc_score(y_tr, _sig(oof_tr_logits))), 4),
        "fold_val_aurocs": fold_val_aucs,
        "validation": _clean_m(val_m),
        "held_out_test": _clean_m(te_m),
        "logreg_reference_test": _clean_m(lr_m),
        "feature_drift_train_vs_test": {r["Feature"]: r for r in drift_rows},
    },
}

out_json = os.path.join(OUTPUT_DIR, "predictive_model.json")
with open(out_json, "w") as f:
    json.dump(model_artifact, f, separators=(",", ":"))
print(f"Exported scaler.json & predictive_model.json ({os.path.getsize(out_json)/1024:.1f} KB)")
print("5-Fold PLE-ResNet Val Summary :", model_artifact["monitoring_metrics"]["validation"])
print("5-Fold PLE-ResNet Test Summary:", model_artifact["monitoring_metrics"]["held_out_test"])


## Summary & Production Deployment Guide
- **Real-Time Alignment & Honest Class Imbalance Handling:** Trained chronologically using strictly causal `<= hour t` bedside history and `t_sepsis - 6h` labels (`70% Train / 15% Val / 15% Test`, zero patient overlap). `Train` addresses the natural `~43:1` hourly ICU imbalance mathematically via inverse-prevalence `pos_weight` + Pairwise AUROC Ranking Loss while `Val` and `Test` preserve untouched incidence so PPV and Platt calibration remain realistic.
- **Deep Tabular Neural Performance & Quantile Calibration:** Piecewise-Linear Quantile Encodings (`PLE`), missingness masks, `signed-log1p` compression, **5-Fold `PLE-ResNet` bagging**, and Validation Platt Calibration (`ECE < 0.005`, `Brier ~0.020`) outperform linear, tree, and `SwiGLUNet` baselines across Hourly (`AUROC ~0.82` on full Kaggle cohort) and Patient-Episode Precision-Recall (`~4.9x` baseline lift).
- **Two-Tier Bedside Alert Discipline:** Locks both a high-sensitivity passive **`Watchlist`** threshold (Utility-optimal nursing screen) and a **`Critical Actionable Alert`** threshold budgeted at **`<= 5.0 FP/100h` (`>= 95%` Hourly Specificity)** on Validation, paired with **`2-of-3h` Sustained Episode Confirmation** (`~90-96%` patient specificity, `~30-44%` patient PPV) to eliminate alarm fatigue.
